# RAG fallback notebook (Google Colab)

Use this if the laptop dies, the venv breaks, or Streamlit will not start.
It is the same pipeline as `app.py` with the Streamlit layer removed.

**Run order:** Setup -> Keys -> Upload -> Index -> Ask.

Open it at https://colab.research.google.com -> GitHub tab -> paste this repo URL.

## 1. Setup (~2 minutes)

In [ ]:
!git clone -q https://github.com/Geetarthjain15/HCL-Hackathon.git
%cd HCL-Hackathon
!pip install -q -r requirements.txt
!apt-get install -qq tesseract-ocr > /dev/null
print('setup done')

## 2. Keys

`getpass` keeps the key out of the saved notebook, so it is safe to commit this file.

In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API key (gsk_...): ')
os.environ['GEMINI_API_KEY'] = getpass('Gemini API key (optional, press Enter to skip): ')
print('keys set')

## 3. Pipeline

In [ ]:
# The whole pipeline lives in the rag package, so there is nothing to
# redefine here - it is the same code the Streamlit app runs.
import rag
from rag.chunking import split_documents
from rag.loaders import load_bytes, load_dir
from rag.pipeline import ask
from rag.store import Store

print('rag package loaded')

## 4. Upload documents

In [ ]:
from google.colab import files

uploaded = files.upload()   # or skip this cell to use the bundled data/

docs = []
for name, data in uploaded.items():
    docs += load_bytes(name, data)
if not docs:
    docs = load_dir('data')

store = Store.build(split_documents(docs))
print(f'indexed {len(store)} chunks from {len(store.sources)} sources')

## 5. Ask

In [ ]:
answer = ask('How long do I have to claim a damaged item?', store)

print(answer.text)
print()
print('sources:', ', '.join(answer.sources))
print('provider:', answer.provider, '| grounded:', answer.grounded)

## 6. Score the pipeline

Runs the same eval set as `python -m rag eval` on the laptop.

In [ ]:
# Same scoring the laptop runs, so numbers are comparable.
from rag.evaluate import format_report, run

results, summary = run(store, answers=True)
print(format_report(results, summary))